In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 10.8 MB/s eta 0:00:00


In [ ]:
from Bio import Entrez

# E-postanızı NCBI'a tanıtmanız gerekir
Entrez.email = "barbarbilge@gmail.com"

# 1. TP53 geni için arama yapalım (İnsan için)
# Term: Arama terimi, db: veritabanı (gene)
handle = Entrez.esearch(db="gene", term="TP53[Gene Name] AND Homo sapiens[Organism]")
record = Entrez.read(handle)
handle.close()

# Arama sonucunda bulunan ilk ID'yi alalım
if record["IdList"]:
    gene_id = record["IdList"][0]
    print(f"Bulunan Gene ID: {gene_id}")

    # 2. Bu ID ile gen detaylarını çekelim
    fetch_handle = Entrez.efetch(db="gene", id=gene_id, retmode="xml")
    gene_record = Entrez.read(fetch_handle)
    fetch_handle.close()

    # Gen özetini yazdıralım
    gene_info = gene_record[0]
    summary = gene_info.get('Entrezgene_summary', 'Özet bulunamadı.')
    print(f"\nGen Özeti:\n{summary}")
else:
    print("Gen bulunamadı.")

Bulunan Gene ID: 7157

Gen Özeti:
This gene encodes a tumor suppressor protein containing transcriptional activation, DNA binding, and oligomerization domains. The encoded protein responds to diverse cellular stresses to regulate expression of target genes, thereby inducing cell cycle arrest, apoptosis, senescence, DNA repair, or changes in metabolism. Mutations in this gene are associated with a variety of human cancers, including hereditary cancers such as Li-Fraumeni syndrome. Alternative splicing of this gene and the use of alternate promoters result in multiple transcript variants and isoforms. Additional isoforms have also been shown to result from the use of alternate translation initiation codons from identical transcript variants (PMIDs: 12032546, 20937277). [provided by RefSeq, Dec 2016]


### Kodun Detaylı Açıklaması:

1.  **`Entrez.email`**: NCBI üzerinden veri çekerken kendinizi tanıtmanız gerekir. Bu, bir sorun oluştuğunda NCBI'ın size ulaşabilmesi içindir.
2.  **`Entrez.esearch`**: Belirtilen veritabanında (`db="gene"`) arama yapar. Burada TP53 gen adını ve insan (`Homo sapiens`) organizmasını filtreleyerek doğru ID'yi alıyoruz.
3.  **`Entrez.read`**: NCBI'dan gelen XML formatındaki yanıtı Python tarafından okunabilir bir sözlük (dictionary) yapısına dönüştürür.
4.  **`record["IdList"]`**: Aramanızla eşleşen NCBI kimlik numaralarının listesini tutar.
5.  **`Entrez.efetch`**: Belirli bir ID'ye sahip kaydın tüm içeriğini getirir. `retmode="xml"` ile veriyi XML formatında istiyoruz ki Biopython bunu kolayca işleyebilsin.
6.  **`Entrezgene_summary`**: TP53 geninin işlevi hakkında tıbbi ve biyolojik özeti içeren alandır.

In [ ]:
from Bio import Entrez, SeqIO
from Bio.Seq import Seq

# NCBI'dan TP53 nükleotit dizisini (RefSeq) çekelim
# NM_000546.6 insan TP53 transkript varyantı 1 için yaygın bir erişim numarasıdır
accession_id = "NM_000546.6"
handle = Entrez.efetch(db="nucleotide", id=accession_id, rettype="fasta", retmode="text")
seq_record = SeqIO.read(handle, "fasta")
handle.close()

# 1. İlk 100 nükleotidi FASTA formatında yazdır
print("--- TP53 İlk 100 Nükleotid (FASTA) ---")
print(f">{seq_record.id} [first 100 bp]\n{seq_record.seq[:100]}")

# 2. Toplam nükleotid sayısını yazdır
print(f"\nToplam Nükleotid Sayısı: {len(seq_record.seq)}")

--- TP53 İlk 100 Nükleotid (FASTA) ---
>NM_000546.6 [first 100 bp]
CTCAAAAGTCTAGAGCCACCGTCCAGGGAGCAGGTAGCTGCTGGGCTCCGGGGACACTTTGCGTTCGGGCTGGGAGCGTGCTTTCCACGACGGTGACACG

Toplam Nükleotid Sayısı: 2512


In [ ]:
# 3. Transkripsiyon (DNA -> mRNA)
dna_seq = seq_record.seq
mrna_seq = dna_seq.transcribe()

print("--- mRNA Dizisi (İlk 50 baz) ---")
print(mrna_seq[:50])

# 4. Translasyon (mRNA -> Protein)
# 'to_stop=True' ilk durdurma kodonunda durmasını sağlar
protein_seq = mrna_seq.translate(to_stop=True)

print("\n--- Protein Dizisi (İlk 50 amino asit) ---")
print(protein_seq[:50])
print(f"\nProtein Zinciri Uzunluğu: {len(protein_seq)} amino asit")

--- mRNA Dizisi (İlk 50 baz) ---
CUCAAAAGUCUAGAGCCACCGUCCAGGGAGCAGGUAGCUGCUGGGCUCCG

--- Protein Dizisi (İlk 50 amino asit) ---
LKSLEPPSREQVAAGLRGHFAFGLGACFPRR

Protein Zinciri Uzunluğu: 31 amino asit


/usr/local/lib/python3.13/dist-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


### Yapılan İşlemlerin Açıklaması:

1.  **`SeqIO.read`**: NCBI'dan gelen FASTA verisini bir `SeqRecord` nesnesine dönüştürür. Bu nesne üzerinden `.seq` ile ham diziyi alabiliriz.
2.  **`transcribe()`**: DNA dizisindeki Timin (T) bazlarını Urasil (U) ile değiştirerek mRNA molekülünü simüle eder.
3.  **`translate()`**: mRNA dizisini üçerli gruplar (kodonlar) halinde okuyarak standart biyolojik kod tablosuna göre amino asit zincirine dönüştürür. `to_stop=True` parametresi, dizideki ilk 'Stop' kodonuna gelindiğinde işlemi durdurur.

In [ ]:
from Bio import Align

# 1. Yaban tipi (wild-type) protein dizisini alalım
wt_protein = str(protein_seq)

# 2. R273H mutasyonunu simüle edelim
# (Dizi uzunluğu yeterli değilse son karakteri değiştirelim veya örnekleyelim)
# İnsan TP53'te 273. pozisyon Arjinin (R) nükleotit/protein eşleşmesine göre belirlenir.
mutation_pos = 272 # 0 tabanlı indeks

if len(wt_protein) > mutation_pos:
    mutant_list = list(wt_protein)
    old_aa = mutant_list[mutation_pos]
    mutant_list[mutation_pos] = "H" # Histidin ile değiştir
    mutant_protein = "".join(mutant_list)
    print(f"Mutasyon uygulandı: Pozisyon {mutation_pos+1}, {old_aa} -> H")
else:
    # Dizi kısa ise örnek bir mutasyon yapalım (örneğin 10. pozisyon)
    mutation_pos = 10
    mutant_list = list(wt_protein)
    old_aa = mutant_list[mutation_pos]
    mutant_list[mutation_pos] = "H"
    mutant_protein = "".join(mutant_list)
    print(f"Dizi kısa olduğu için {mutation_pos+1}. pozisyonda mutasyon yapıldı: {old_aa} -> H")

# 3. Bio.Align kullanarak hizalama yapalım
aligner = Align.PairwiseAligner()
aligner.mode = 'global'
alignments = aligner.align(wt_protein, mutant_protein)

# 4. En iyi hizalamayı görselleştirelim
best_alignment = alignments[0]
print("\n--- Protein Hizalaması (WT vs Mutant) ---")
print(best_alignment)

Dizi kısa olduğu için 11. pozisyonda mutasyon yapıldı: Q -> H

--- Protein Hizalaması (WT vs Mutant) ---
target            0 LKSLEPPSREQVAAGLRGHFAFGLGACFPRR 31
                  0 ||||||||||.|||||||||||||||||||| 31
query             0 LKSLEPPSREHVAAGLRGHFAFGLGACFPRR 31



### Yapılan İşlemlerin Açıklaması:

1.  **Mutasyon Simülasyonu**: Python'da dizeler (string) değiştirilemez (immutable) olduğu için diziyi önce bir listeye çevirdik, ilgili indeksteki karakteri değiştirdik ve tekrar dize haline getirdik.
2.  **`Bio.Align.PairwiseAligner`**: İki dizi arasındaki benzerliği hesaplamak için kullanılan modern Biopython aracıdır. `global` modu, dizilerin tamamını uçtan uca hizalar.
3.  **Görsel Çıktı**: Hizalama çıktısında iki dizi arasındaki dikey çizgiler (`|`) eşleşen amino asitleri, boşluklar veya farklılıklar ise mutasyonun gerçekleştiği noktayı gösterir.